# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [20]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [21]:
LITE_MODE = True

In [22]:
username = "ed-donner"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

In [23]:
items[2].id

In [24]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [25]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [26]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [27]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


Title: Schlage F59 Interior Knob with Deadbolt (Oil Rubbed Bronze, Interior Half)  
Category: Hardware & Door Hardware  
Brand: Schlage  
Description: A durable, oil‑rubbed bronze interior knob paired with a deadbolt for enhanced security.  
Details: Features easy installation, a 4″ minimum center‑to‑center door prep requirement, and a lifetime mechanical and finish warranty.

Input tokens: 446
Output tokens: 97
Cost: 0.009 cents


In [28]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


### Product
Title: Schlage Interior Security Handle Set
Category: Home Security
Brand: Schlage
Description: A secure and stylish interior door handle set with a deadbolt for added protection.
Details: Features 100% solid precision engineering with a lifetime mechanical and finish warranty.

Input tokens: 406
Output tokens: 58
Cost: 0.000 cents


In [29]:
MODEL = "gpt-4o-mini"


In [30]:
def make_jsonl(item):
    body = {
        "model": MODEL, 
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT}, 
            {"role": "user", "content": item.full}
        ], 
        "reasoning_effort": "low" # Note: 'reasoning_effort' is for o1/o3 models. Remove this key for gpt-4o-mini or gpt-3.5
    }
    
    # Clean up reasoning_effort if not using reasoning models
    if "o1" not in MODEL and "o3" not in MODEL:
        body.pop("reasoning_effort", None)

    line = {
        "custom_id": str(item.id), 
        "method": "POST", 
        "url": "/v1/chat/completions", 
        "body": body
    }
    return json.dumps(line)

In [31]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [32]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gpt-4o-mini", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piece model.\'

In [33]:
def make_file(start, end, filename):
    with open(filename, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [34]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [35]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [36]:
from openai import OpenAI

# Initialize OpenAI Client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# 1. Upload the file
with open("jsonl/0_1000.jsonl", "rb") as f:
    batch_input_file = client.files.create(
        file=f,
        purpose="batch"
    )

print(f"File uploaded. ID: {batch_input_file.id}")

# 2. Create the Batch
batch_response = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
      "description": "price-is-right-preprocessing"
    }
)

print(f"Batch submitted! Batch ID: {batch_response.id}")

File uploaded. ID: file-Mc7PWbv6gfDihyBrDL9WQh
Batch submitted! Batch ID: batch_696c7aa4a9b481908e40fa7a0c5a91ca


In [37]:
import time

# Poll the batch status
batch_id = batch_response.id

print(f"Checking status for {batch_id}...")

while True:
    batch_status = client.batches.retrieve(batch_id)
    print(f"Status: {batch_status.status}")
    
    if batch_status.status in ["completed", "failed", "cancelled"]:
        break
    
    time.sleep(10) # Wait 10 seconds before checking again

if batch_status.status == "completed":
    print("Batch complete! Downloading results...")
    
    # Retrieve the output file content
    file_response = client.files.content(batch_status.output_file_id)
    
    # Save to disk
    with open("jsonl/batch_results.jsonl", "w") as f:
        f.write(file_response.text)
        
    print("Results saved to jsonl/batch_results.jsonl")

elif batch_status.status == "failed":
    print("Batch failed.")
    print(batch_status.errors)

Checking status for batch_696c7aa4a9b481908e40fa7a0c5a91ca...
Status: validating
Status: validating
Status: validating
Status: validating
Status: validating
Status: validating
Status: validating
Status: in_progress
Status: in_progress
Status: in_progress
Status: in_progress
Status: in_progress
Status: in_progress
Status: in_progress
Status: in_progress
Status: in_progress
Status: in_progress
Status: finalizing
Status: finalizing
Status: finalizing
Status: finalizing
Status: finalizing
Status: finalizing
Status: completed
Batch complete! Downloading results...
Results saved to jsonl/batch_results.jsonl


In [38]:
import json

# Load the results back into the Item objects
print("Mapping results to items...")
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        try:
            data = json.loads(line)
            # The custom_id we sent was the item's ID
            item_id = int(data["custom_id"])
            
            # Extract the summary from the OpenAI response structure
            summary = data["response"]["body"]["choices"][0]["message"]["content"]
            
            # Update the item in memory
            items[item_id].summary = summary
        except Exception as e:
            print(f"Skipping line due to error: {e}")

# Verify it worked
print(f"Item 0 Summary: {items[0].summary}")

Mapping results to items...
Item 0 Summary: Title: Schlage Andover Interior Knob and Deadbolt 
Category: Home Hardware
Brand: Schlage
Description: A stylish and secure oil-rubbed bronze interior knob paired with a deadbolt for enhanced safety.
Details: Features easy installation and a lifetime warranty for both mechanical and finish durability.


In [39]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [41]:
# 1. Fill missing summaries with empty strings to ensure consistent data types
print("Standardizing empty summaries...")
for item in items:
    if item.summary is None:
        item.summary = ""

print("Done. Now retrying upload...")

# 2. Now run your push code again
username = "ed-donner" # Ensure this matches your HF username
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)
    
    # Also push lite version if in full mode
    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

Standardizing empty summaries...
Done. Now retrying upload...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

HfHubHTTPError: (Request ID: Root=1-696c7f34-102a2e353ea57d106d5a1194;6f20c6ea-6181-432c-be67-55933092aea2)

403 Forbidden: You have read access but not the required permissions for this operation.
Cannot access content at: https://huggingface.co/datasets/ed-donner/items_lite.git/info/lfs/objects/batch.
Make sure your token has the correct permissions.

## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
